# Picoからのシリアル通信を受け取る
- 加速度センサ

In [5]:
import serial
import array
import numpy as np

from IPython.display import clear_output

In [6]:
import numpy as np
import datetime
import time
# Poseの各ランドマークを保存
class body1():
    def __init__(self, dirName="motion"):
        self.dirName = dirName
        self.fileName = "motion"
        self.vec_size = 3
        
        # 以下は触らない
        self.isRecord = False
        self.data = np.empty((0,self.vec_size,3), dtype=np.float32)
        self.time_ary = []
        
        
    def record1(self, values):
        if self.isRecord == False:
            return False
        
        data = np.empty((self.vec_size, 3), dtype=np.float32)
        for i in range(self.vec_size):
            data[i] = values[i]

        self.data = np.vstack([self.data, np.array([data])])
        self.time_ary.append(time.perf_counter())
        return True
    
    
    def lerp_data(self, interval=0.01):
        new_time_ary = []
        new_data_ary = np.empty((0,self.vec_size,3), dtype=np.float32)
        
        np_time_array = np.array(self.time_ary)
        for t in np.arange(0, self.time_ary[-1], interval):
            upper_idx = min(np.where(np_time_array > t)[0])
            lower_idx = upper_idx - 1
            if (lower_idx < 0):
                continue
            elif (upper_idx >= len(np_time_array)):
                break
            lower_data = self.data[lower_idx]
            upper_data = self.data[upper_idx]
            diff = upper_data - lower_data
            time_ratio = (t - np_time_array[lower_idx]) / (np_time_array[upper_idx] - np_time_array[lower_idx])
            new_data = lower_data + diff * time_ratio
            
            new_data_ary = np.vstack([new_data_ary, np.array([new_data])])
            new_time_ary.append(t)
            
        self.data = new_data_ary
        self.time_ary = new_time_ary
        return

    
    def save(self, data_label, isLerp=False):
        # numpy行列を保存
        data = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        filename_data = self.dirName + "/" + self.fileName + data
        #np.save(filename_data, self.data)
        # CSVに保存
        print(len(self.time_ary))
        for i in range(len(self.time_ary) - 1):
            self.time_ary[i+1] = self.time_ary[i+1] - self.time_ary[0]
        self.time_ary[0] = 0.0
        label1 = np.array(data_label * 3)
        label1 = label1.reshape([3, self.vec_size]).reshape(-1, order='F')
        label2 = ['X', 'Y', 'Z'] * self.vec_size
        label = "Frame,Time," + ",".join(label1) + "\n" + ",," + ",".join(label2)
        
        if (isLerp):
            self.lerp_data()
            
        print(filename_data)
        print(self.data.shape)
        print(self.data)
            
        frame_ary = list(range(len(self.time_ary)))
        csv_data = self.data.reshape([self.data.shape[0], 3*self.vec_size])
        csv_data = np.insert(csv_data, 0, self.time_ary, axis=1)
        csv_data = np.insert(csv_data, 0, frame_ary, axis=1)
        np.savetxt(filename_data + ".csv", csv_data, delimiter=',', 
                   header=label, comments='',
                   fmt='%f')

        
    def start_stop(self):
        if self.isRecord == False:
            self.isRecord = True
            clear_output(wait=True)
            print("Start")
            return
        else:
            label = ["Accl", "Gyro", "Mag"]
            self.save(label, True)
            print("Recorded")
            self.data = np.empty((0,self.vec_size,3), dtype=np.float32)
            self.time_ary = []
            self.isRecord = False
            return

In [7]:
import cv2
import time

cap = cv2.imshow("window", np.zeros((100,100)))

ser = serial.Serial("COM3", 115200)
learnData = body1("flight")

def read_acceralation():
    ser.write((0).to_bytes(1, 'little'))
    message = ser.readline()
    message = message.strip().decode('UTF-8')
    message_split = message.split(',')
    value_array = np.float_(message_split)
    value_array = value_array.reshape((3, 3))
    return value_array


while (True):
    value_array = read_acceralation()
    
    #result_sagittal = mediapipe2.loop()
    key = cv2.waitKey(1) & 0xFF
    if key == 27:    # Escで終了
        break;
    if key == ord(' '):    # Spaceキーで録画・終了
        clear_output(wait=True)
        learnData.start_stop()

    learnData.record1(value_array)

    
cv2.destroyAllWindows()
ser.close()

167
flight/motion20230423_181758
(264, 3, 3)
[[[ 7.69999981e-01  4.90000010e-01  1.00500002e+01]
  [-1.54999995e+00  2.59999990e+00  8.70000005e-01]
  [-6.80000000e+01  6.90000000e+01 -3.60000000e+01]]

 [[ 7.69999981e-01  4.90000010e-01  1.00500002e+01]
  [ 9.95016098e-02  4.29108572e+00 -1.09831297e+00]
  [-6.80000000e+01  6.90000000e+01 -3.60000000e+01]]

 [[ 7.69999981e-01  4.90000010e-01  1.00500002e+01]
  [ 1.18111634e+00  5.06837320e+00 -2.03738594e+00]
  [-6.69360123e+01  6.90000000e+01 -3.67093277e+01]]

 ...

 [[-8.51902291e-02  9.01940918e+00  5.40919828e+00]
  [-2.68366838e+00 -1.18557072e+00  4.36253405e+00]
  [-1.14000000e+02  6.70000000e+01  6.50000000e+01]]

 [[-2.99999993e-02  9.06999969e+00  5.40000010e+00]
  [-3.08511615e+00 -1.02335906e+00  3.64736795e+00]
  [-1.14335915e+02  6.73359146e+01  6.51679535e+01]]

 [[-2.99999993e-02  9.06999969e+00  5.40000010e+00]
  [-3.14143968e+00 -1.03587544e+00  2.37070036e+00]
  [-1.15587547e+02  6.85875473e+01  6.57937775e+01]]]
R

In [4]:
cv2.destroyAllWindows()
ser.close()